# TP3 - Travail à faire

In [5]:
import pandas as pd
import time

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif, f_classif

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## 1. DATA LOADING 

In [ ]:
CSV_FILENAME = "E-commerce_Dataset.csv" 
LABEL_COLUMN = "label"
TEXT_COLUMN = "text"

df = pd.read_csv(CSV_FILENAME, header=None, names=[LABEL_COLUMN, TEXT_COLUMN])

df = df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])
print(f"Total documents loaded: {len(df)}")

X = df[TEXT_COLUMN]
y_raw = df[LABEL_COLUMN]

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Total documents loaded: 50424


## 2. TEXT VECTORIZATION (TF-IDF)

In [8]:
vectorizer = TfidfVectorizer(max_features=10000)
X_train_tfidf = vectorizer.fit_transform(X_train_raw)
X_test_tfidf = vectorizer.transform(X_test_raw)

## 3. DEFINE THE EXPERIMENT PARAMETERS

In [10]:
K_FEATURES = 1000

feature_selectors = {
    "Chi-Square": SelectKBest(score_func=chi2, k=K_FEATURES),
    "Mutual Information": SelectKBest(score_func=mutual_info_classif, k=K_FEATURES),
    "Information Gain": SelectKBest(score_func=mutual_info_classif, k=K_FEATURES),
    "Fisher's Score": SelectKBest(score_func=f_classif, k=K_FEATURES)
}

classifiers = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": LinearSVC(random_state=42),
    "XGBoost": XGBClassifier(
        tree_method="hist", 
        device="cuda",
        use_label_encoder=False, 
        eval_metric='mlogloss',
        random_state=42
    )
}

## 4. THE COMPARATIVE STUDY

In [ ]:
results = []

for fs_name, selector in feature_selectors.items():
    print(f"\n[{fs_name}] Selecting top {K_FEATURES} features...")
    start_time = time.time()
    
    # Apply Feature Selection
    X_train_selected = selector.fit_transform(X_train_tfidf, y_train)
    X_test_selected = selector.transform(X_test_tfidf)
    
    print(f"Feature selection completed in {time.time() - start_time:.2f} seconds.")
    
    for clf_name, clf in classifiers.items():
        print(f"  -> Training {clf_name}...")
        
        # Train the model
        clf.fit(X_train_selected, y_train)
        
        # Make predictions
        y_pred = clf.predict(X_test_selected)
        
        # Calculate Metrics (using average='macro' for multi-class classification)
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
        rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
        
        # Save results
        results.append({
            "Feature Selection": fs_name,
            "Classifier": clf_name,
            "Accuracy": round(acc, 4),
            "Precision": round(prec, 4),
            "Recall": round(rec, 4),
            "F1-Score": round(f1, 4)
        })


[Chi-Square] Selecting top 1000 features...
Feature selection completed in 0.23 seconds.
  -> Training Decision Tree...


## 5. DISPLAY RESULTS

In [ ]:
results_df = pd.DataFrame(results)

print("\n==========================================")
print(" FINAL COMPARATIVE STUDY RESULTS")
print("==========================================\n")
# Sort by F1-Score to see the best performing combinations
results_df = results_df.sort_values(by="F1-Score", ascending=False).reset_index(drop=True)
print(results_df.to_string())

# Save to CSV to easily copy into your lab report
results_df.to_csv("tp3_classification_results.csv", index=False)
print("\nResults successfully saved to 'tp3_classification_results.csv'")